In [2]:
# ================================
# Cell 1: Import Libraries
# ================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


In [3]:
# ================================
# Cell 2: Load Data
# ================================

df = pd.read_csv('../data/olx_used_cars.csv')

print("✅ Data loaded successfully!")
print(f"\n📊 Shape: {df.shape}")
print(f"📋 Columns: {df.columns.tolist()}")
print(f"\n🔍 First 5 rows:")
df.head()

✅ Data loaded successfully!

📊 Shape: (9179, 17)
📋 Columns: [' ', 'Price', 'Make', 'Model', 'Year', "KM's driven", 'Fuel', 'Registration city', 'Car documents', 'Assembly', 'Transmission', 'Condition', 'Seller Location', 'Description', 'Car Features', "Images URL's", 'Car Profile']

🔍 First 5 rows:


,,Price,Make,Model,Year,KM's driven,Fuel,Registration city,Car documents,Assembly,Transmission,Condition,Seller Location,Description,Car Features,Images URL's,Car Profile
0,0,4190000,Toyota,Passo,2021,54000,Petrol,Unregistered,Original,Imported,Automatic,Used,"Airline Avenue, Islamabad","it's 2021 model fresh import, perfect engine s...","ABS, Air Bags, AM/FM Radio, CD Player, Cassett...",['https://images.olx.com.pk/thumbnails/4039460...,https://www.olx.com.pk/item/fresh-import-passo...
1,1,1300000,Suzuki,Ravi,2018,95000,Petrol,Karachi,Original,Local,Manual,Used,"Kahuta, Rawalpindi",Suzuki ravi 2018 col,AM/FM Radio,['https://images.olx.com.pk/thumbnails/4102504...,https://www.olx.com.pk/item/suzuki-ravi-iid-10...
2,2,800000,Suzuki,Bolan,2015,50000,Petrol,Karachi,Original,Local,Manual,Used,"Lyari Expressway, Karachi",Suzuki bolan model 2015 reg 2022 ghadi me koi ...,Rear speakers,['https://images.olx.com.pk/thumbnails/4139520...,https://www.olx.com.pk/item/suzuki-bolan-2015-...
3,3,2155000,Daihatsu,Move,2013,94000,Petrol,Lahore,Original,Imported,Automatic,Used,"New Amir Town, Lahore",Move Push start \nHome Used car. \nModel 2013....,"ABS, Air Bags, Air Conditioning, Alloy Rims, A...",['https://images.olx.com.pk/thumbnails/3865337...,https://www.olx.com.pk/item/diahatsu-move-2013...
4,4,1440000,Suzuki,Swift,2011,126544,Petrol,Karachi,Original,Local,Manual,Used,"Shadman 2, Karachi",Suzuki Swift DLX\nAbS Break 100%\nengine 100%\...,"ABS, Air Conditioning, Alloy Rims, AM/FM Radio...",['https://images.olx.com.pk/thumbnails/4143509...,https://www.olx.com.pk/item/suzuki-swift-dlx-2...


In [3]:
# ================================
# Cell 3: Convert Prices to SAR & Create Categories
# ================================

# 1. التحويل من روبية باكستانية إلى ريال سعودي
# 1 PKR = 0.01355 SAR (سعر تقريبي)
PKR_TO_SAR = 0.01355

df['Price_SAR'] = df['Price'] * PKR_TO_SAR

print("=" * 60)
print("💰 أسعار السيارات بعد التحويل إلى الريال السعودي")
print("=" * 60)
print(f"\nأقل سعر:  {df['Price_SAR'].min():,.0f} ريال سعودي")
print(f"أعلى سعر: {df['Price_SAR'].max():,.0f} ريال سعودي")
print(f"متوسط:    {df['Price_SAR'].mean():,.0f} ريال سعودي")

# 2. تقسيم الأسعار إلى 3 فئات
df['Price_Category'] = pd.qcut(
    df['Price_SAR'], 
    q=3, 
    labels=['Cheap', 'Medium', 'Expensive']
)

print("\n" + "=" * 60)
print("📊 توزيع فئات الأسعار")
print("=" * 60)
print(df['Price_Category'].value_counts())

print("\n" + "=" * 60)
print("📋 عينة من البيانات")
print("=" * 60)
df[['Make', 'Model', 'Year', 'Price', 'Price_SAR', 'Price_Category']].head(10)

💰 أسعار السيارات بعد التحويل إلى الريال السعودي

أقل سعر:  2,507 ريال سعودي
أعلى سعر: 67,750 ريال سعودي
متوسط:    27,599 ريال سعودي

📊 توزيع فئات الأسعار
Cheap        3061
Medium       3060
Expensive    3058
Name: Price_Category, dtype: int64

📋 عينة من البيانات


,Make,Model,Year,Price,Price_SAR,Price_Category
0,Toyota,Passo,2021,4190000,56774.50,Expensive
1,Suzuki,Ravi,2018,1300000,17615.00,Medium
2,Suzuki,Bolan,2015,800000,10840.00,Cheap
3,Daihatsu,Move,2013,2155000,29200.25,Medium
4,Suzuki,Swift,2011,1440000,19512.00,Medium
5,Suzuki,Wagon R,2020,2830000,38346.50,Expensive
6,Daihatsu,Mira,2013,2150000,29132.50,Medium
7,Suzuki,Bolan,2016,1050000,14227.50,Cheap
8,Honda,City IDSI,2004,1230000,16666.50,Cheap
9,Suzuki,Cultus VXR,2008,950000,12872.50,Cheap


In [4]:
# ================================
# Cell 4: Prepare Features for Modeling
# ================================

# 1. تحديد الميزات (Features) والهدف
# نستخدم الأعمدة المفيدة فقط
feature_cols = ['Make', 'Model', 'Year', "KM's driven", 'Fuel', 'Transmission']

# التحقق من وجودها
available_cols = [c for c in feature_cols if c in df.columns]
print("✅ الأعمدة المتاحة:", available_cols)

# 2. حذف الصفوف التي فيها قيم مفقودة (في هذه الأعمدة والهدف)
df_clean = df[available_cols + ['Price_SAR', 'Price_Category']].dropna()

print(f"\n📊 عدد الصفوف بعد التنظيف: {len(df_clean)}")

# 3. تحويل النصوص إلى أرقام (Label Encoding)
from sklearn.preprocessing import LabelEncoder

encoders = {}
for col in available_cols:
    if df_clean[col].dtype == 'object':
        le = LabelEncoder()
        df_clean[col] = le.fit_transform(df_clean[col].astype(str))
        encoders[col] = le
        print(f"✅ تم ترميز عمود: {col}")

# 4. تحديد X و y
X = df_clean[available_cols]
y_reg = df_clean['Price_SAR']          # للانحدار
y_clf = df_clean['Price_Category']     # للتصنيف

print(f"\n✅ X shape: {X.shape}")
print(f"✅ y_reg shape: {y_reg.shape}")
print(f"✅ y_clf shape: {y_clf.shape}")

✅ الأعمدة المتاحة: ['Make', 'Model', 'Year', "KM's driven", 'Fuel', 'Transmission']

📊 عدد الصفوف بعد التنظيف: 9179
✅ تم ترميز عمود: Make
✅ تم ترميز عمود: Model
✅ تم ترميز عمود: Fuel
✅ تم ترميز عمود: Transmission

✅ X shape: (9179, 6)
✅ y_reg shape: (9179,)
✅ y_clf shape: (9179,)


In [9]:
# ================================
# Cell 5: Train Regression Model
# ================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

# تدريب النموذج
print("⏳ جاري تدريب نموذج الانحدار...")
reg_model = RandomForestRegressor(n_estimators=100, random_state=42)
reg_model.fit(X_train, y_train)

# التنبؤ والتقييم
y_pred = reg_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n" + "=" * 60)
print("📈 نتائج نموذج الانحدار (Regression)")
print("=" * 60)
print(f"MAE (متوسط الخطأ): {mae:,.0f} ريال سعودي")
print(f"R² Score: {r2:.4f}")
print("=" * 60)

⏳ جاري تدريب نموذج الانحدار...

📈 نتائج نموذج الانحدار (Regression)
MAE (متوسط الخطأ): 44,260 ريال سعودي
R² Score: 0.8348


In [10]:
# ================================
# Cell 6: Train Classification Model
# ================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# تقسيم البيانات
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_clf, test_size=0.2, random_state=42
)

# تدريب النموذج
print("⏳ جاري تدريب نموذج التصنيف...")
clf_model = RandomForestClassifier(n_estimators=100, random_state=42)
clf_model.fit(X_train_c, y_train_c)

# التنبؤ والتقييم
y_pred_c = clf_model.predict(X_test_c)

accuracy = accuracy_score(y_test_c, y_pred_c)

print("\n" + "=" * 60)
print("🎯 نتائج نموذج التصنيف (Classification)")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f}")
print(f"\n📋 تقرير التصنيف الكامل:")
print(classification_report(y_test_c, y_pred_c))
print("=" * 60)

⏳ جاري تدريب نموذج التصنيف...

🎯 نتائج نموذج التصنيف (Classification)
Accuracy: 0.6150

📋 تقرير التصنيف الكامل:
              precision    recall  f1-score   support

       Cheap       0.86      0.68      0.76      2157
   Expensive       0.54      0.60      0.57      2135
      Medium       0.51      0.57      0.54      2144

    accuracy                           0.61      6436
   macro avg       0.64      0.61      0.62      6436
weighted avg       0.64      0.61      0.62      6436



In [7]:
# ================================
# Cell 7: Save Models
# ================================

import joblib
import os

# إنشاء مجلد models إذا لم يكن موجوداً
os.makedirs('../models', exist_ok=True)

# حفظ النماذج
joblib.dump(reg_model, '../models/regression_model.pkl')
joblib.dump(clf_model, '../models/classification_model.pkl')
joblib.dump(encoders, '../models/encoders.pkl')
joblib.dump(available_cols, '../models/features.pkl')

print("✅ تم حفظ النماذج بنجاح!")
print("\n📁 الملفات المحفوظة في مجلد models/:")
print("   - regression_model.pkl      (نموذج الانحدار)")
print("   - classification_model.pkl  (نموذج التصنيف)")
print("   - encoders.pkl              (مشفّرات النصوص)")
print("   - features.pkl              (أسماء الميزات)")
print("\n🎉 مشروع تنقيب البيانات اكتمل!")

✅ تم حفظ النماذج بنجاح!

📁 الملفات المحفوظة في مجلد models/:
   - regression_model.pkl      (نموذج الانحدار)
   - classification_model.pkl  (نموذج التصنيف)
   - encoders.pkl              (مشفّرات النصوص)
   - features.pkl              (أسماء الميزات)

🎉 مشروع تنقيب البيانات اكتمل!


In [8]:
# ================================
# Cell 8: Save Results to CSV
# ================================

# 1. حفظ البيانات الكاملة مع التحويلات
df.to_csv('../data/car_data_processed.csv', index=False)
print("✅ تم حفظ البيانات المعالجة: car_data_processed.csv")

# 2. حفظ نتائج التنبؤ (الانحدار)
results_reg = pd.DataFrame({
    'Actual_Price_SAR': y_test.values,
    'Predicted_Price_SAR': y_pred,
    'Difference_SAR': y_test.values - y_pred
})
results_reg.to_csv('../data/regression_results.csv', index=False)
print("✅ تم حفظ نتائج الانحدار: regression_results.csv")

# 3. حفظ نتائج التصنيف
results_clf = pd.DataFrame({
    'Actual_Category': y_test_c.values,
    'Predicted_Category': y_pred_c
})
results_clf.to_csv('../data/classification_results.csv', index=False)
print("✅ تم حفظ نتائج التصنيف: classification_results.csv")

# 4. حفظ مقاييس الأداء (للتقرير)
metrics = pd.DataFrame({
    'Model': ['Regression (Random Forest)', 'Classification (Random Forest)'],
    'Metric': ['R² Score', 'Accuracy'],
    'Value': [round(r2, 4), round(accuracy, 4)]
})
metrics.to_csv('../data/model_metrics.csv', index=False)
print("✅ تم حفظ مقاييس الأداء: model_metrics.csv")

# 5. حفظ ملخص الفئات
category_summary = df['Price_Category'].value_counts().reset_index()
category_summary.columns = ['Category', 'Count']
category_summary.to_csv('../data/category_summary.csv', index=False)
print("✅ تم حفظ ملخص الفئات: category_summary.csv")

print("\n" + "=" * 60)
print("🎉 تم حفظ جميع النتائج بنجاح!")
print("=" * 60)
print("\n📁 الملفات المحفوظة في مجلد data/:")
print("   1. car_data_processed.csv      (البيانات الكاملة)")
print("   2. regression_results.csv      (نتائج الانحدار)")
print("   3. classification_results.csv  (نتائج التصنيف)")
print("   4. model_metrics.csv           (مقاييس الأداء)")
print("   5. category_summary.csv        (ملخص الفئات)")

✅ تم حفظ البيانات المعالجة: car_data_processed.csv
✅ تم حفظ نتائج الانحدار: regression_results.csv
✅ تم حفظ نتائج التصنيف: classification_results.csv
✅ تم حفظ مقاييس الأداء: model_metrics.csv
✅ تم حفظ ملخص الفئات: category_summary.csv

🎉 تم حفظ جميع النتائج بنجاح!

📁 الملفات المحفوظة في مجلد data/:
   1. car_data_processed.csv      (البيانات الكاملة)
   2. regression_results.csv      (نتائج الانحدار)
   3. classification_results.csv  (نتائج التصنيف)
   4. model_metrics.csv           (مقاييس الأداء)
   5. category_summary.csv        (ملخص الفئات)


In [4]:
# ================================
# Cell 9: Load & Merge New Dataset
# ================================

# 1. تحميل الملف الجديد
df_new = pd.read_csv('../data/Updated_Car_Sales_Data.csv')

print("=" * 60)
print("📊 معلومات الملف الجديد")
print("=" * 60)
print(f"Shape: {df_new.shape}")
print(f"Columns: {df_new.columns.tolist()}")

# 2. إعادة تسمية الأعمدة لتتوافق مع البيانات القديمة
df_new = df_new.rename(columns={
    'Car Make': 'Make',
    'Car Model': 'Model',
    'Mileage': "KM's driven",
    'Fuel Type': 'Fuel'
})

# 3. إضافة عمود Condition و Accident (موجودان بالفعل)
# لكن نتأكد من وجودهما
print(f"\n✅ Condition موجود: {'Condition' in df_new.columns}")
print(f"✅ Accident موجود: {'Accident' in df_new.columns}")

# 4. عرض الأعمدة بعد إعادة التسمية
print(f"\n📋 الأعمدة الجديدة: {df_new.columns.tolist()}")

# 5. نظرة على البيانات
df_new.head()

📊 معلومات الملف الجديد
Shape: (23000, 11)
Columns: ['Car Make', 'Car Model', 'Year', 'Mileage', 'Price', 'Fuel Type', 'Color', 'Transmission', 'Options/Features', 'Condition', 'Accident']

✅ Condition موجود: True
✅ Accident موجود: True

📋 الأعمدة الجديدة: ['Make', 'Model', 'Year', "KM's driven", 'Price', 'Fuel', 'Color', 'Transmission', 'Options/Features', 'Condition', 'Accident']


,Make,Model,Year,KM's driven,Price,Fuel,Color,Transmission,Options/Features,Condition,Accident
0,Hyundai,Tucson,2010,52554,44143.820,Hybrid,Black,Automatic,Heated Seats,Used,No
1,Land Rover,Range Rover,2016,115056,25414.060,Diesel,Silver,Manual,GPS,Used,No
2,Honda,Accord,2022,18044,28262.872,Electric,Yellow,Manual,"Bluetooth, Leather Seats, Backup Camera, GPS",Like New,Yes
3,Kia,Soul,2011,79251,28415.848,Hybrid,Orange,Manual,"Backup Camera, GPS, Heated Seats, Leather Seats",New,Yes
4,Volkswagen,Golf,2022,40975,31509.792,Electric,Silver,Automatic,"Leather Seats, Remote Start, Sunroof, Heated S...",New,Yes


In [5]:
# ================================
# Cell 10: Merge Both Datasets
# ================================

# 1. تحديد الأعمدة المشتركة
common_cols = ['Make', 'Model', 'Year', "KM's driven", 'Price', 'Fuel', 'Transmission']
new_extra_cols = ['Condition', 'Accident']

# 2. تجهيز df_new (نأخذ الأعمدة المطلوبة فقط)
df_new_subset = df_new[common_cols + new_extra_cols].copy()

# 3. تجهيز df الأصلي (إضافة Condition و Accident بقيم افتراضية)
df_old = df[common_cols].copy()
df_old['Condition'] = 'Used'   # افتراضي
df_old['Accident'] = 'No'      # افتراضي

# 4. دمج الملفين
df_merged = pd.concat([df_old, df_new_subset], ignore_index=True)

print("=" * 60)
print("📊 بعد الدمج")
print("=" * 60)
print(f"عدد السيارات: {len(df_merged):,}")
print(f"الأعمدة: {df_merged.columns.tolist()}")
print(f"\nتوزيع Condition:")
print(df_merged['Condition'].value_counts())
print(f"\nتوزيع Accident:")
print(df_merged['Accident'].value_counts())

df_merged.head()

📊 بعد الدمج
عدد السيارات: 32,179
الأعمدة: ['Make', 'Model', 'Year', "KM's driven", 'Price', 'Fuel', 'Transmission', 'Condition', 'Accident']

توزيع Condition:
Used        16742
New          7721
Like New     7716
Name: Condition, dtype: int64

توزيع Accident:
Yes    18566
No     13613
Name: Accident, dtype: int64


,Make,Model,Year,KM's driven,Price,Fuel,Transmission,Condition,Accident
0,Toyota,Passo,2021,54000,4190000.0,Petrol,Automatic,Used,No
1,Suzuki,Ravi,2018,95000,1300000.0,Petrol,Manual,Used,No
2,Suzuki,Bolan,2015,50000,800000.0,Petrol,Manual,Used,No
3,Daihatsu,Move,2013,94000,2155000.0,Petrol,Automatic,Used,No
4,Suzuki,Swift,2011,126544,1440000.0,Petrol,Manual,Used,No


In [6]:
# ================================
# Cell 11: Preprocess Merged Data
# ================================

# 1. نسخة جديدة من البيانات المدمجة
df_final = df_merged.copy()

# 2. تحويل Price إلى SAR (البيانات الجديدة بالدولار، القديمة بالروبية)
# نفترض أن الأسعار الجديدة بالدولار → نحولها إلى SAR
USD_TO_SAR = 3.75
PKR_TO_SAR = 0.01355

# نحدد الصفوف الجديدة (التي فيها Condition من الملف الجديد)
# الصفوف القديمة: Price صغير (بالروبية)، الصفوف الجديدة: Price كبير (بالدولار)
df_final['Price_SAR'] = df_final['Price'].apply(
    lambda x: x * PKR_TO_SAR if x < 100000 else x * USD_TO_SAR
)

print("=" * 60)
print("📊 بعد التحويل إلى SAR")
print("=" * 60)
print(f"أقل سعر: {df_final['Price_SAR'].min():,.0f} SAR")
print(f"أعلى سعر: {df_final['Price_SAR'].max():,.0f} SAR")
print(f"متوسط: {df_final['Price_SAR'].mean():,.0f} SAR")

# 3. إنشاء الفئات السعرية
df_final['Price_Category'] = pd.qcut(
    df_final['Price_SAR'],
    q=3,
    labels=['Cheap', 'Medium', 'Expensive']
)

print(f"\nتوزيع الفئات:")
print(df_final['Price_Category'].value_counts())

# 4. تحويل Condition و Accident إلى أرقام
df_final['Condition'] = df_final['Condition'].map({
    'Used': 0, 'Like New': 1, 'New': 2
}).fillna(0).astype(int)

df_final['Accident'] = df_final['Accident'].map({
    'No': 0, 'Yes': 1
}).fillna(0).astype(int)

print(f"\n✅ Condition (بعد التحويل): {df_final['Condition'].unique()}")
print(f"✅ Accident (بعد التحويل): {df_final['Accident'].unique()}")

df_final.head()

📊 بعد التحويل إلى SAR
أقل سعر: 54 SAR
أعلى سعر: 18,750,000 SAR
متوسط: 2,249,103 SAR

توزيع الفئات:
Cheap        10727
Medium       10726
Expensive    10726
Name: Price_Category, dtype: int64

✅ Condition (بعد التحويل): [0 1 2]
✅ Accident (بعد التحويل): [0 1]


,Make,Model,Year,KM's driven,Price,Fuel,Transmission,Condition,Accident,Price_SAR,Price_Category
0,Toyota,Passo,2021,54000,4190000.0,Petrol,Automatic,0,0,15712500.0,Expensive
1,Suzuki,Ravi,2018,95000,1300000.0,Petrol,Manual,0,0,4875000.0,Expensive
2,Suzuki,Bolan,2015,50000,800000.0,Petrol,Manual,0,0,3000000.0,Expensive
3,Daihatsu,Move,2013,94000,2155000.0,Petrol,Automatic,0,0,8081250.0,Expensive
4,Suzuki,Swift,2011,126544,1440000.0,Petrol,Manual,0,0,5400000.0,Expensive


In [7]:
# ================================
# Cell 12: Fix Price Conversion
# ================================

# نستخدم عمود Condition للتمييز
# لكن للأسف، البيانات القديمة أيضاً Used
# الحل: نستخدم Accident كمؤشر
# البيانات القديمة كلها Accident = No (افتراضي)
# لكن البيانات الجديدة فيها Yes و No معاً

# الحل الأفضل: نعيد الدمج بطريقة مختلفة
# نضيف عمود "source" قبل الدمج

# نعيد بناء البيانات من الصفر
df_old_clean = df[['Make', 'Model', 'Year', "KM's driven", 'Price', 'Fuel', 'Transmission']].copy()
df_old_clean['Condition'] = 'Used'
df_old_clean['Accident'] = 'No'
df_old_clean['Source'] = 'Pakistan'

df_new_clean = df_new[['Make', 'Model', 'Year', "KM's driven", 'Price', 'Fuel', 'Transmission', 'Condition', 'Accident']].copy()
df_new_clean['Source'] = 'Global'

df_final = pd.concat([df_old_clean, df_new_clean], ignore_index=True)

# الآن نحول الأسعار حسب المصدر
PKR_TO_SAR = 0.01355
USD_TO_SAR = 3.75

df_final['Price_SAR'] = df_final.apply(
    lambda row: row['Price'] * PKR_TO_SAR if row['Source'] == 'Pakistan' 
                else row['Price'] * USD_TO_SAR,
    axis=1
)

print("=" * 60)
print("📊 بعد التحويل الصحيح")
print("=" * 60)
print(f"أقل سعر: {df_final['Price_SAR'].min():,.0f} SAR")
print(f"أعلى سعر: {df_final['Price_SAR'].max():,.0f} SAR")
print(f"متوسط: {df_final['Price_SAR'].mean():,.0f} SAR")

print(f"\nتوزيع المصدر:")
print(df_final['Source'].value_counts())

# الفئات السعرية
df_final['Price_Category'] = pd.qcut(
    df_final['Price_SAR'],
    q=3,
    labels=['Cheap', 'Medium', 'Expensive']
)

print(f"\nتوزيع الفئات:")
print(df_final['Price_Category'].value_counts())

df_final.head()

📊 بعد التحويل الصحيح
أقل سعر: 2,507 SAR
أعلى سعر: 1,124,708 SAR
متوسط: 131,269 SAR

توزيع المصدر:
Global      23000
Pakistan     9179
Name: Source, dtype: int64

توزيع الفئات:
Cheap        10727
Medium       10726
Expensive    10726
Name: Price_Category, dtype: int64


,Make,Model,Year,KM's driven,Price,Fuel,Transmission,Condition,Accident,Source,Price_SAR,Price_Category
0,Toyota,Passo,2021,54000,4190000.0,Petrol,Automatic,Used,No,Pakistan,56774.50,Medium
1,Suzuki,Ravi,2018,95000,1300000.0,Petrol,Manual,Used,No,Pakistan,17615.00,Cheap
2,Suzuki,Bolan,2015,50000,800000.0,Petrol,Manual,Used,No,Pakistan,10840.00,Cheap
3,Daihatsu,Move,2013,94000,2155000.0,Petrol,Automatic,Used,No,Pakistan,29200.25,Cheap
4,Suzuki,Swift,2011,126544,1440000.0,Petrol,Manual,Used,No,Pakistan,19512.00,Cheap


In [8]:
# ================================
# Cell 13: Prepare Final Features
# ================================

from sklearn.preprocessing import LabelEncoder

# 1. اختيار الميزات (بما فيها الجديدة)
feature_cols = ['Make', 'Model', 'Year', "KM's driven", 'Fuel', 'Transmission', 'Condition', 'Accident']

# 2. حذف الصفوف التي فيها قيم مفقودة
df_model = df_final[feature_cols + ['Price_SAR', 'Price_Category']].dropna()

print(f"عدد الصفوف: {len(df_model):,}")

# 3. ترميز النصوص
encoders = {}
for col in feature_cols:
    if df_model[col].dtype == 'object':
        le = LabelEncoder()
        df_model[col] = le.fit_transform(df_model[col].astype(str))
        encoders[col] = le
        print(f"✅ {col}: {len(le.classes_)} فئة")

# 4. تحضير X و y
X = df_model[feature_cols]
y_reg = df_model['Price_SAR']
y_clf = df_model['Price_Category']

print(f"\nX shape: {X.shape}")
print(f"y_reg shape: {y_reg.shape}")
print(f"y_clf shape: {y_clf.shape}")

عدد الصفوف: 32,179
✅ Make: 35 فئة
✅ Model: 177 فئة
✅ Fuel: 6 فئة
✅ Transmission: 2 فئة
✅ Condition: 3 فئة
✅ Accident: 2 فئة

X shape: (32179, 8)
y_reg shape: (32179,)
y_clf shape: (32179,)


In [11]:
# ================================
# Cell 14: Train on Pakistan Data + New Features
# ================================

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, classification_report

# 1. نأخذ البيانات الباكستانية فقط
df_pk = df_final[df_final['Source'] == 'Pakistan'].copy()

print("=" * 60)
print("📊 البيانات الباكستانية مع الميزات الجديدة")
print("=" * 60)
print(f"عدد السيارات: {len(df_pk):,}")

# 2. الميزات (بما فيها Condition و Accident)
feature_cols = ['Make', 'Model', 'Year', "KM's driven", 'Fuel', 'Transmission', 'Condition', 'Accident']

# 3. تحويل Condition و Accident إلى أرقام
df_pk['Condition'] = df_pk['Condition'].map({
    'Used': 0, 'Like New': 1, 'New': 2
}).fillna(0).astype(int)

df_pk['Accident'] = df_pk['Accident'].map({
    'No': 0, 'Yes': 1
}).fillna(0).astype(int)

# 4. ترميز النصوص
encoders_pk = {}
for col in feature_cols:
    if df_pk[col].dtype == 'object':
        le = LabelEncoder()
        df_pk[col] = le.fit_transform(df_pk[col].astype(str))
        encoders_pk[col] = le

# 5. تحديد X و y
X_pk = df_pk[feature_cols]
y_reg_pk = df_pk['Price_SAR']
y_clf_pk = df_pk['Price_Category']

print(f"X shape: {X_pk.shape}")

# ================================
# 6. تدريب نموذج الانحدار
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X_pk, y_reg_pk, test_size=0.2, random_state=42
)

reg_model_pk = RandomForestRegressor(n_estimators=100, random_state=42)
reg_model_pk.fit(X_train, y_train)

y_pred = reg_model_pk.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n" + "=" * 60)
print("📈 نتائج الانحدار (باكستان + ميزات جديدة)")
print("=" * 60)
print(f"MAE: {mae:,.0f} SAR")
print(f"R² Score: {r2:.4f}")

# ================================
# 7. تدريب نموذج التصنيف
# ================================
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_pk, y_clf_pk, test_size=0.2, random_state=42
)

clf_model_pk = RandomForestClassifier(n_estimators=100, random_state=42)
clf_model_pk.fit(X_train_c, y_train_c)

y_pred_c = clf_model_pk.predict(X_test_c)
accuracy = accuracy_score(y_test_c, y_pred_c)

print("\n" + "=" * 60)
print("🎯 نتائج التصنيف (باكستان + ميزات جديدة)")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f}")
print(f"\nتقرير التصنيف:")
print(classification_report(y_test_c, y_pred_c))

📊 البيانات الباكستانية مع الميزات الجديدة
عدد السيارات: 9,179
X shape: (9179, 8)

📈 نتائج الانحدار (باكستان + ميزات جديدة)
MAE: 2,597 SAR
R² Score: 0.9411

🎯 نتائج التصنيف (باكستان + ميزات جديدة)
Accuracy: 0.9641

تقرير التصنيف:
              precision    recall  f1-score   support

       Cheap       0.97      0.98      0.98      1450
      Medium       0.92      0.90      0.91       386

    accuracy                           0.96      1836
   macro avg       0.95      0.94      0.95      1836
weighted avg       0.96      0.96      0.96      1836



In [12]:
# ================================
# Cell 15: Save New Models
# ================================

import joblib

# حفظ النماذج الجديدة
joblib.dump(reg_model_pk, '../models/regression_model_final.pkl')
joblib.dump(clf_model_pk, '../models/classification_model_final.pkl')
joblib.dump(encoders_pk, '../models/encoders_final.pkl')
joblib.dump(feature_cols, '../models/features_final.pkl')

print("✅ تم حفظ النماذج الجديدة بنجاح!")
print("\n📁 الملفات المحفوظة:")
print("   - regression_model_final.pkl")
print("   - classification_model_final.pkl")
print("   - encoders_final.pkl")
print("   - features_final.pkl")

# ملخص النتائج النهائية
print("\n" + "=" * 60)
print("🏆 ملخص النتائج النهائية")
print("=" * 60)
print(f"📈 الانحدار: R² = {r2:.4f}  |  MAE = {mae:,.0f} SAR")
print(f"🎯 التصنيف: Accuracy = {accuracy:.4f}")
print(f"📊 عدد السيارات: {len(df_pk):,}")
print(f"🔢 عدد الميزات: {len(feature_cols)}")
print("=" * 60)
print("\n🎉 المشروع اكتمل!")

✅ تم حفظ النماذج الجديدة بنجاح!

📁 الملفات المحفوظة:
   - regression_model_final.pkl
   - classification_model_final.pkl
   - encoders_final.pkl
   - features_final.pkl

🏆 ملخص النتائج النهائية
📈 الانحدار: R² = 0.9411  |  MAE = 2,597 SAR
🎯 التصنيف: Accuracy = 0.9641
📊 عدد السيارات: 9,179
🔢 عدد الميزات: 8

🎉 المشروع اكتمل!


In [13]:
# ================================
# Cell 16: Verify with Cross-Validation
# ================================

from sklearn.model_selection import cross_val_score

print("=" * 60)
print("🔍 التحقق المتقاطع (5-Fold Cross-Validation)")
print("=" * 60)

# 1. للتصنيف
cv_clf = cross_val_score(clf_model_pk, X_pk, y_clf_pk, cv=5, scoring='accuracy')
print(f"\n📊 التصنيف:")
print(f"   النتائج: {[f'{s:.4f}' for s in cv_clf]}")
print(f"   المتوسط: {cv_clf.mean():.4f}")
print(f"   الانحراف: {cv_clf.std():.4f}")

# 2. للانحدار
cv_reg = cross_val_score(reg_model_pk, X_pk, y_reg_pk, cv=5, scoring='r2')
print(f"\n📈 الانحدار:")
print(f"   النتائج: {[f'{s:.4f}' for s in cv_reg]}")
print(f"   المتوسط: {cv_reg.mean():.4f}")
print(f"   الانحراف: {cv_reg.std():.4f}")

# 3. تحليل النتيجة
print("\n" + "=" * 60)
print("🎯 التفسير")
print("=" * 60)

if cv_clf.mean() > 0.90 and cv_clf.std() < 0.05:
    print("✅ النتيجة حقيقية - النموذج مستقر")
elif cv_clf.mean() > 0.90 and cv_clf.std() >= 0.05:
    print("⚠️ النتيجة جيدة لكن فيها تباين - راجعي البيانات")
else:
    print("❌ النتيجة ضعيفة - تحتاج تحسين")

🔍 التحقق المتقاطع (5-Fold Cross-Validation)

📊 التصنيف:
   النتائج: ['0.9602', '0.9684', '0.9662', '0.9635', '0.9520']
   المتوسط: 0.9621
   الانحراف: 0.0057

📈 الانحدار:
   النتائج: ['0.9403', '0.9464', '0.9439', '0.9405', '0.9403']
   المتوسط: 0.9423
   الانحراف: 0.0025

🎯 التفسير
✅ النتيجة حقيقية - النموذج مستقر
